# Blind test transcription

Transcribes the 659 blind-test clips and writes `qxp.tsv` in the organisers'
format: no header, two tab-separated columns, one line per audio file, in the
order of their template.

The checkpoint scores CER 3.07 / WER 19.44 on scripted dev and 18.69 / 36.19 on
spontaneous dev.

Decoding is the manual CTC collapse; `tok.decode(skip_special_tokens)` inflates
error by about 8.5 WER. `max_sec` is 30: eleven spontaneous test clips run past
20 s.


In [ ]:
import os, json, csv, re, time
from pathlib import Path
from collections import Counter

WORK = Path('/kaggle/working')
CFG  = dict(model='facebook/w2v-bert-2.0', batch=4, max_sec=30.0)

TEST = next((p.parent for p in Path('/kaggle/input').rglob('qxp_test_dataset.tsv')), None)
assert TEST is not None, 'attach qxp-blind-test'
AUD = {p.name: p for p in TEST.rglob('*.wav')}
meta = list(csv.DictReader(open(TEST/'qxp_test_dataset.tsv', encoding='utf-8'), delimiter='\t'))
order = [l.split('\t')[0] for l in (TEST/'qxp.tsv').read_text(encoding='utf-8').splitlines() if l.strip()]
print(f'{len(AUD)} wavs | {len(meta)} metadata rows | {len(order)} template rows')
assert len(order) == 659, len(order)
assert set(order) <= set(AUD), 'template names missing from audio'
print(Counter(r['type'] for r in meta))

In [ ]:
!pip -q install 'transformers>=4.44' 'torchaudio' 'jiwer' 2>&1 | tail -1
import torch, transformers
print(transformers.__version__, torch.cuda.get_device_name(0))

In [ ]:
import torch, torchaudio
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2BertForCTC, SeamlessM4TFeatureExtractor

VOCAB_SRC = next(p for p in Path('/kaggle/input').rglob('vocab.json')
                 if (p.parent/'model.safetensors').exists())
vocab = json.loads(VOCAB_SRC.read_text(encoding='utf-8'))
INV = {v: k for k, v in vocab.items()}
BLANK = vocab['[PAD]']
feat = SeamlessM4TFeatureExtractor.from_pretrained(CFG['model'])
print('vocab', len(vocab), 'from', VOCAB_SRC)

def load_audio(path):
    w, sr = torchaudio.load(str(path))
    if w.shape[0] > 1: w = w.mean(0, keepdim=True)
    if sr != 16000: w = torchaudio.functional.resample(w, sr, 16000)
    return w.squeeze(0)[: int(CFG['max_sec']*16000)].numpy()

class Clips(Dataset):
    def __init__(self, names): self.names = names
    def __len__(self): return len(self.names)
    def __getitem__(self, i):
        y = load_audio(AUD[self.names[i]])
        return feat(y, sampling_rate=16000, return_tensors='pt').input_features[0]

def collate(batch):
    T = max(x.shape[0] for x in batch)
    X = torch.zeros(len(batch), T, batch[0].shape[1])
    mask = torch.zeros(len(batch), T, dtype=torch.long)
    for i, x in enumerate(batch):
        X[i, :x.shape[0]] = x; mask[i, :x.shape[0]] = 1
    return X, mask

def collapse(seq):
    prev, chars = None, []
    for t in seq:
        if t != prev and t != BLANK:
            chars.append(INV.get(t, ''))
        prev = t
    return re.sub(r'\s+', ' ', ''.join(chars).replace('|', ' ')).strip()

In [ ]:
from huggingface_hub import snapshot_download, HfApi
from kaggle_secrets import UserSecretsClient

tok_hf = UserSecretsClient().get_secret('HF_TOKEN')
user = HfApi(token=tok_hf).whoami()['name']
snapshot_download(f'{user}/qxp-vs2', allow_patterns=['vs2/*'],
                  local_dir=str(WORK/'hub'), token=tok_hf)
CKPT = WORK/'hub'/'vs2'
assert (CKPT/'config.json').exists(), 'vs2 checkpoint not found on the Hub'
print('submitting with', CKPT)
model = Wav2Vec2BertForCTC.from_pretrained(CKPT).cuda().eval()

In [ ]:
PART = WORK/'preds.jsonl'
done = {}
if PART.exists():
    for line in PART.read_text(encoding='utf-8').splitlines():
        if line.strip():
            d = json.loads(line); done[d['file']] = d['hyp']
todo = [n for n in order if n not in done]
print(f'{len(order)} clips | {len(done)} done | {len(todo)} to go', flush=True)

t0 = time.time(); i = 0
with torch.no_grad(), PART.open('a', encoding='utf-8') as ph:
    dl = DataLoader(Clips(todo), batch_size=CFG['batch'], collate_fn=collate, num_workers=2)
    for X, mask in dl:
        with torch.amp.autocast('cuda', dtype=torch.float16):
            logits = model(input_features=X.cuda(), attention_mask=mask.cuda()).logits
        out_len = model._get_feat_extract_output_lengths(mask.sum(-1)).tolist()
        for k, ids in enumerate(logits.argmax(-1).cpu().tolist()):
            hyp = collapse(ids[: out_len[k]])
            done[todo[i]] = hyp
            ph.write(json.dumps({'file': todo[i], 'hyp': hyp}, ensure_ascii=False) + '\n')
            i += 1
        ph.flush(); os.fsync(ph.fileno())
        if i % 100 < CFG['batch']:
            print(f'  {i}/{len(todo)}  {(time.time()-t0)/60:.1f}m', flush=True)
print(f'decoded {i} clips in {(time.time()-t0)/60:.1f} min')

In [ ]:
import unicodedata as ud, zipfile
out = WORK/'qxp.tsv'
with out.open('w', encoding='utf-8', newline='\n') as fh:
    for n in order:
        fh.write(f"{n}\t{ud.normalize('NFC', done[n])}\n")

rows = [l.split('\t') for l in out.read_text(encoding='utf-8').splitlines()]
empty = [r[0] for r in rows if len(r) < 2 or not r[1].strip()]
assert len(rows) == 659, len(rows)
assert all(len(r) == 2 for r in rows), 'not two columns'
assert [r[0] for r in rows] == order, 'order differs from the template'
print(f'wrote {out}: {len(rows)} lines, {len(empty)} empty')
if empty: print('EMPTY:', empty[:10])

z = WORK/'qxp_submission.zip'
with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(out, arcname='qxp.tsv')
print('zipped ->', z)

bytype = {}
for r in meta:
    bytype.setdefault(r['type'], []).append(done[r['file_name']])
for k, v in bytype.items():
    w = [len(x.split()) for x in v]
    print(f'{k:12s} n={len(v):4d}  mean words/clip {sum(w)/len(w):5.1f}  empty {sum(1 for x in v if not x)}')
print()
for r in meta[:3]:
    print(f"  {r['type']:11s} {done[r['file_name']][:78]}")